# TaaS Studio — Colab T4 Quick Launch

**Default mode:** run the notebook from top to bottom. No ModelProfile upload, provider key, or manual model endpoint is required to launch Studio.

The notebook installs the TaaS package, detects available hardware, creates a temporary access token, starts the web application inside this Colab VM, and opens it through Colab's port proxy.

**Current capability boundary:** this launch milestone provides the existing TaaS discovery workspace. Hugging Face model management, chat-agent execution, MCP invocation, streaming traces and persistent run history are subsequent implementation milestones. Do not interpret the discovery workspace as an operational agent.

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, secrets
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "65be37263c19f33865f1a5628c7e24e9e474fe4d"
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
APP_HOST, APP_PORT = "127.0.0.1", 8000

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
assert (WORKDIR / "pyproject.toml").is_file()
assert (WORKDIR / "src" / "taas").is_dir()
print("TaaS revision:", actual)

## Hardware detection

A T4 is recommended for the future local-model workflow. The current discovery core itself runs on CPU. If no GPU is allocated, Studio can still launch in discovery mode.

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; CPU discovery mode.")
print("Python:", sys.version.split()[0])

## Install and verify TaaS Studio

This cell installs into the active Colab Python interpreter and explicitly verifies that the package is importable. If it fails, the notebook stops here with the pip diagnostic.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[studio]"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(install.stdout[-4000:])
if install.returncode:
    print(install.stderr[-4000:])
    raise RuntimeError("TaaS Studio installation failed")
subprocess.run(
    [sys.executable, "-c", "import taas, fastapi, uvicorn; import taas.studio.api; print('TaaS Studio imports OK:', taas.__file__)"],
    cwd=WORKDIR, check=True
)

## Launch Studio

An ephemeral API token is generated automatically for this runtime. Copy it into the Studio token field when using the discovery API. It is not committed or persisted. No calibration profile is required: Studio starts in **uncalibrated strict mode**, where unmeasured model-native capabilities are not assumed.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
print("Temporary Studio API token (copy into Studio; do not share):")
print(api_token)

if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    studio_process.wait(timeout=10)

studio_log = open("/content/taas-studio.log", "w")
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=40):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HOST}:{APP_PORT}/health"))
print("Readiness:", wait_http(f"http://{APP_HOST}:{APP_PORT}/ready"))

## Open TaaS Studio

Run this cell and open the link. Keep the Colab runtime connected while using Studio.

In [ ]:
from google.colab import output
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))

## Stop Studio

Run this cell when finished. Colab storage and runtime memory are temporary.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    studio_process.wait(timeout=10)
if "studio_log" in globals():
    studio_log.close()
print("TaaS Studio stopped.")